# matmul-2d — ex1: predict matmul output shape and verify with @

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `matmul-2d`. Running the final beacon cell reports progress against the `Numpy: matmul 2-D` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: matmul 2-D` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`matmul-2d`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "matmul-2d"
DD_SUBTOPIC = "Numpy: matmul 2-D"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Numpy matmul 2-D — quick refresher

For 2-D tensors, matrix multiplication has the shape rule:

```
(M, K) @ (K, N) → (M, N)
```

The **inner** dimensions must match (both equal `K`); they're contracted (summed) away. The **outer** dimensions become the output shape (`M` from the left, `N` from the right).

**Python `@` operator.** `a @ b` dispatches to `torch.matmul(a, b)` (or `numpy.matmul`). For pure 2-D inputs this is identical to `a.matmul(b)`, `t.mm(a, b)`, and `einops.einsum(a, b, 'm k, k n -> m n')`.

**Per-element formula.** Each output entry is the dot product of one row of `a` with one column of `b`:

```
out[i, j] = sum_k a[i, k] * b[k, j]
```

**Common mistakes.**
- `(M, K) @ (N, K)` shape-errors — the second matrix needs a transpose first (`a @ b.T` gives `(M, N)`).
- Confusing `matmul` with `mul` (elementwise) — `a * b` and `a @ b` do totally different things; the former needs broadcasting-compatible shapes, the latter needs the (K, K) inner match.

**Higher dims.** For batched inputs, `(..., M, K) @ (..., K, N) → (..., M, N)` — the leading batch dims broadcast. The 2-D rule is the base case.

### Exercise 1 — predict matmul output shape and verify with @

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the `(M, K) @ (K, N) -> (M, N)` matmul shape rule by predicting valid/invalid pairs and dispatching `@` against concrete tensors.
> Keywords: matmul, at-operator, shape-rule, inner-dim-match
> ```

**KCs targeted:** `matmul-2d-shape-rule`, `at-operator-dispatches-matmul`

Implement TWO functions:

**1.** `ex1_matmul_outshape(shape_a, shape_b)` — pure shape logic, no tensors. Inputs are 2-tuples (each shape is `(rows, cols)`). Return:
- The output shape tuple `(M, N)` if the inner dimensions match.
- `None` if they don't.

```
ex1_matmul_outshape((3, 4), (4, 5)) -> (3, 5)
ex1_matmul_outshape((3, 4), (5, 4)) -> None    # inner dims 4 != 5
ex1_matmul_outshape((2, 2), (2, 2)) -> (2, 2)  # square
```

**2.** `ex1_matmul(a, b)` — given two 2-D tensors, return their matrix product using the `@` operator. (This is a one-liner; the point is to dispatch via `@`, not via `t.mm` or `einops.einsum`.)

**Why two functions in one drill.** The drill targets BOTH KCs: predicting the output shape WITHOUT running the op (the interview / debug skill), and dispatching the op via `@` (the idiomatic Python form). They share the same shape-rule but exercise it at different levels — one symbolic, one concrete.

**No higher-dim cases.** This drill is strictly 2-D. Batched matmul has its own broadcasting rules — out of scope here.

In [ ]:
def ex1_matmul_outshape(shape_a: tuple, shape_b: tuple):
    m, k1 = shape_a
    k2, n = shape_b
    if k1 != k2:
        return None
    return (m, n)


def ex1_matmul(a: Tensor, b: Tensor) -> Tensor:
    return a @ b


<details><summary>Solution</summary>

```python
def ex1_matmul_outshape(shape_a: tuple, shape_b: tuple):
    m, k1 = shape_a
    k2, n = shape_b
    if k1 != k2:
        return None
    return (m, n)


def ex1_matmul(a: Tensor, b: Tensor) -> Tensor:
    return a @ b
```

**Why `@` is the idiomatic form.** PEP 465 added the `@` operator specifically for matrix multiplication. It dispatches via `__matmul__` to `torch.matmul` (for tensors), `numpy.matmul` (for ndarrays), or any custom implementation via `__matmul__` / `__rmatmul__`. Equivalents include `t.mm(a, b)` (2-D only), `t.matmul(a, b)` (any dims), and `einops.einsum(a, b, 'm k, k n -> m n')`.

**Why the shape rule has THIS form.** The matmul definition is `out[i, j] = sum_k a[i, k] * b[k, j]`. The `k` index must RANGE over the same values on both sides (it's the same sum index) — so `a.shape[1]` and `b.shape[0]` must be equal. The free indices `i, j` survive into the output as rows and columns.

**Quick mnemonic.** Write the shapes left-to-right with their inner dimensions touching: `(M, K)(K, N)`. The middle `K`s cancel like algebra — what's left is `(M, N)`.

**Why this is its own atom (not subsumed by einsum-contraction).** Einsum is more general but more verbose. For pure 2-D matmul, `@` is shorter, faster to read, and what PyTorch's `nn.Linear` calls under the hood (via `F.linear → addmm`). Knowing when to reach for `@` vs `einsum` is its own skill.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()